<a href="https://colab.research.google.com/github/pujaroy280/DATA612/blob/main/DATA_612_Project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 3: Matrix Factorization methods**


Puja Roy

Spring 2026


## **Introduction**

Online platforms use recommendation systems to help users find things they like. These recommendation systems look at what users do and what they like to figure out what these users might like. Companies like Netflix, Amazon and Spotify use these recommendation systems to make the users happy.

One way to make these recommendation systems is to use a method called matrix factorization. This matrix factorization method takes a list of users and the things these users like. It breaks this list down into lists that show hidden patterns in what these users like. For example it finds out that some users like movies with an actor or that some movies have similar themes.

The goal of this project was to use SVD on these user ratings to make the data smaller guess the ratings and give movie recommendations to the users. We wanted to see if SVD and MovieLens 100K could help us make a movie recommendation system for the users. It worked well for the users.

The recommendation systems and the matrix factorization technique used in this project can be applied to areas like music recommendations or product recommendations. This can help the users discover things they will like such, as music artists or products using these recommendation systems.

## **Step 1: Import Libraries and Load MovieLens 100K Dataset**

In [5]:
# Import the operating system module for file path operations
import os

# Import zipfile for extracting compressed datasets
import zipfile

# Import Google Drive support for Colab
from google.colab import drive

# Import pandas for data manipulation
import pandas as pd

# Import NumPy for numerical computations
import numpy as np

# Mount Google Drive if you want to save results there
# drive.mount('/content/drive')

# Store the URL of the MovieLens 100K dataset
MOVIELENS_URL = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"

# Define the name of the downloaded zip file
MOVIELENS_ZIP_PATH = "ml-100k.zip"

# Define the extraction folder name
MOVIELENS_EXTRACT_DIR = "ml-100k"

# Check whether the dataset has already been downloaded
if not os.path.exists(MOVIELENS_EXTRACT_DIR):

    # Display download message
    print(f"Downloading MovieLens 100k dataset from {MOVIELENS_URL}...")

    # Download the dataset
    !wget -nc {MOVIELENS_URL}

    # Display extraction message
    print(f"Extracting {MOVIELENS_ZIP_PATH}...")

    # Open the zip file
    with zipfile.ZipFile(MOVIELENS_ZIP_PATH, 'r') as zip_ref:

        # Extract all contents
        zip_ref.extractall(MOVIELENS_EXTRACT_DIR)

    # Confirm extraction
    print("Extraction complete.")

# Build the path to the ratings file
ratings_file_path = os.path.join(
    MOVIELENS_EXTRACT_DIR,
    "ml-100k",
    "u.data"
)

# Load the ratings data into a DataFrame
ratings = pd.read_csv(
    ratings_file_path,
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

# Display the first five rows
print("Sample Ratings Data:")
print(ratings.head())

Sample Ratings Data:
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596


## **Step 2: Create User-Item Matrix**

In [6]:
# Create a matrix with users as rows and movies as columns
user_item_matrix = ratings.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
)

# Display matrix dimensions
print("Matrix Shape:")
print(user_item_matrix.shape)

# Display first few rows
print(user_item_matrix.head())

Matrix Shape:
(943, 1682)
item_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                              ...   
1         5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2         4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5         4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

item_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                              
1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
5         NaN   NaN  

## **Step 3: Handle Missing Values**

In [7]:
# Calculate the average rating for each user
user_means = user_item_matrix.mean(axis=1)

# Subtract each user's average rating from their ratings
ratings_centered = user_item_matrix.sub(
    user_means,
    axis=0
)

# Replace missing values with zero
ratings_centered = ratings_centered.fillna(0)

# Display matrix dimensions
print("Centered Matrix Shape:")
print(ratings_centered.shape)

Centered Matrix Shape:
(943, 1682)


## **Step 4: Perform Singular Value Decomposition (SVD)**

In [10]:
# Import the sparse SVD function
from scipy.sparse.linalg import svds

# Select the number of latent factors
k = 20

# Apply Singular Value Decomposition
# Convert the pandas DataFrame to a NumPy array before passing it to svds
U, sigma, VT = svds(
    ratings_centered.values,
    k=k
)

# Convert singular values into a diagonal matrix
sigma = np.diag(sigma)

# Display matrix dimensions
print("U Shape:", U.shape)
print("Sigma Shape:", sigma.shape)
print("VT Shape:", VT.shape)

U Shape: (943, 20)
Sigma Shape: (20, 20)
VT Shape: (20, 1682)


## **Step 5: Reconstruct Predicted Ratings**

In [11]:
# Multiply U, Sigma, and VT to reconstruct ratings
predicted_ratings = np.dot(
    np.dot(U, sigma),
    VT
)

# Add the user means back
predicted_ratings = predicted_ratings + user_means.values.reshape(-1, 1)

# Convert results into a DataFrame
predicted_df = pd.DataFrame(
    predicted_ratings,
    columns=user_item_matrix.columns,
    index=user_item_matrix.index
)

# Display sample predictions
print(predicted_df.head())

item_id      1         2         3         4         5         6         7     \
user_id                                                                         
1        4.401050  3.558604  3.424559  3.530308  3.270556  3.731752  4.638969   
2        3.558045  3.754272  3.629974  3.724057  3.736624  3.744084  3.706257   
3        2.680490  2.785062  2.724996  2.903132  2.801108  2.804132  2.676728   
4        4.482115  4.351044  4.348538  4.276148  4.293497  4.333113  4.440302   
5        3.701718  2.951298  2.740606  2.998943  2.760134  2.954525  3.800243   

item_id      8         9         10    ...      1673      1674      1675  \
user_id                                ...                                 
1        3.829869  4.633192  3.789770  ...  3.616941  3.610626  3.612294   
2        3.804773  3.750689  3.629998  ...  3.704157  3.709907  3.708734   
3        2.699528  2.680557  2.789657  ...  2.795062  2.796326  2.794810   
4        4.405646  4.385189  4.351163  ...  4.334234

## **Step 6: Generate Recommendations**

In [12]:
# Select a user
user_id = 1

# Get movies already rated by the user
rated_movies = user_item_matrix.loc[user_id]

# Remove missing values
rated_movies = rated_movies.dropna()

# Get predicted ratings
predictions = predicted_df.loc[user_id]

# Remove already-rated movies
predictions = predictions.drop(
    rated_movies.index
)

# Sort by highest predicted rating
recommendations = predictions.sort_values(
    ascending=False
)

# Display top 10 recommendations
print("Top 10 Recommendations:")
print(recommendations.head(10))

Top 10 Recommendations:
item_id
276    5.149023
408    4.608109
483    4.295587
275    4.214321
430    4.193998
508    4.183115
433    4.181367
474    4.180223
511    4.175837
285    4.165756
Name: 1, dtype: float64


## **Step 7: Save Recommendations**

In [13]:
# Convert recommendations into a DataFrame
recommendation_df = pd.DataFrame(
    recommendations.head(10)
)

# Rename the column
recommendation_df.columns = ["Predicted Rating"]

# Save recommendations to CSV
recommendation_df.to_csv(
    "top_10_recommendations.csv"
)

# Display confirmation
print("Recommendations saved successfully.")

Recommendations saved successfully.


## **Discussion**

The objective of this project was to create a Movie Recommender System using a matrix factorization method such as Singular Value Decomposition (SVD) using the MovieLens 100K Dataset. The dataset contains movie ratings from users however, the main problem is that many users have not provided ratings for every movie.

This creates spaces in the user-item matrix. To fix this I adjusted the ratings by subtracting each users rating from their rating. Then I replaced the missing values with zeros. This made it easier for the SVD algorithm to process the matrix and helped the movie recommendation system. SVD breaks down the user-item matrix into matrices. These matrices show hidden patterns in the data called latent features of the MovieLens 100K dataset. This is really useful for the movie recommendation system.

Of comparing every user and movie directly the model uses these latent features. It finds relationships between users and movies. The SVD algorithm and its latent features are crucial to the movie recommendation system. I chose to use twenty latent features for this project. This reduced the size of the data. Kept the important information about the MovieLens 100K dataset. After breaking down the matrix I rebuilt it. This was a step for the movie recommendation system. I used the system to predict ratings for movies that users had not rated. The system recommended movies based on these predicted ratings. That's the goal of the movie recommendation system. One good thing about SVD is that it can discover relationships in the MovieLens 100K dataset.

This is a plus for the movie recommendation system. It also makes predictions quickly once the factorization is done. However, SVD has some limitations. It requires handling of missing values. The latent features are hard to understand because they don't match movie genres or categories. Overall, the results show that SVD is a technique for building recommendation systems like the movie recommendation system. It helps reduce the size of datasets like the MovieLens 100K dataset. SVD is good, for movie recommendations. The movie recommendation system uses SVD. The movie recommendation system and SVD work well with the MovieLens 100K dataset. I think SVD and the movie recommendation system are a combination. The movie recommendation system is based on SVD. I used SVD to make the movie recommendation system. The movie recommendation system is the result of using SVD on the MovieLens 100K dataset.

## **Conclusion**

The goal of this project was to create a system that suggests movies to people. I used a method called Singular Value Decomposition to look at movie ratings from MovieLens 100K and figure out what movies people will like. This method helped me analyze the ratings. I was curious to see if Singular Value Decomposition could really help me find some movies for people. I wanted to test it out. Then I tell them about these movies that I think they will like. To start I made a list of people and a list of movies. I then fixed the missing information in the lists. Next I applied Singular Value Decomposition to this list. Then I put the list together. Looked at the results. I gave some movie ideas to a person I chose to test the system. The system took all the information from MovieLens 100K. Analyzed it. It turned the information into 20 things, which helped it find patterns in what people like. This way it made some guesses about movies that people have not seen yet. The Singular Value Decomposition system was able to look at all the movie ratings and find some things that people like. It looked at the ratings. Made recommendations. This project showed that using Singular Value Decomposition can make a recommendation system better. It takes a lot of information. Makes it simpler. The system still keeps the information. Singular Value Decomposition was really good at figuring out what people like. It was also good at giving people some movie ideas that they might enjoy. The Singular Value Decomposition method is very useful for recommending things to people like movies, from MovieLens 100K. Singular Value Decomposition helps recommend movies to people since it is a great recommender system tool.

## **References**

GroupLens Research. (1998). *MovieLens 100K Dataset*. University of Minnesota. Retrieved from http://files.grouplens.org/datasets/movielens/

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., ... Oliphant, T. E. (2020). Array programming with NumPy. *Nature, 585*(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer, 42*(8), 30–37. https://doi.org/10.1109/MC.2009.263
